# Mise en place d'AWS IoT Core pour l'edge


**Prérequis** : identifiants AWS configurés (`aws configure`)

## 0. Configuration

À adapter si besoin. `THING_NAME` **doit** être égal à `MQTT_CLIENT_ID` de l'edge : la policy n'autorise la connexion que si l'ID client MQTT correspond au nom du thing.

**Compte partagé** : plusieurs membres du groupe utilisent le même compte AWS. Pour ne pas se gêner, chacun a :
- son propre thing / ID client `edge-<utilisateur IAM>` (IoT Core n'accepte qu'une connexion par ID client : deux edges avec le même ID se déconnectent l'un l'autre en boucle) ;
- ses propres noms de machines `coffee-<utilisateur IAM>-01…` (sinon les données des deux simulateurs se mélangent sous les mêmes `device_id`).

La policy, le topic `coffee/+/batch` et la suite du pipeline (règle, Lambda, DynamoDB) restent communs.

In [1]:
import json
import re
import shutil
import time
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

REGION = "us-east-1"
GROUPE_NAME = "GRP10"
POLICY_NAME = "coffee-edge-policy-" + GROUPE_NAME
TOPIC_PREFIX = "coffee-" + GROUPE_NAME
RULE_NAME = "coffee_ingest-" + GROUPE_NAME
LAMBDA_NAME = "coffee-ingestor-" + GROUPE_NAME
ROLE_NAME = "coffee-iot-role-" + GROUPE_NAME
LOG_RETENTION_DAYS = 365
BATCH_MAX_ROWS = 400 

ROOT = Path.cwd().resolve().parents[1]   # racine du dépôt (le notebook est dans AWS/IotCore/)
CERTS = ROOT / "certs"
ENV_FILE = ROOT / ".env"
STATE_FILE = Path.cwd() / ".state.json"  # ARN du certificat etc., ignoré par git

session = boto3.Session(region_name=REGION)
iot = session.client("iot")
iam = session.client("iam")
logs = session.client("logs")
lambda_ = session.client("lambda")
sts = session.client("sts")

identity = sts.get_caller_identity()
ACCOUNT_ID = identity["Account"]
THING_NAME = f"edge-{GROUPE_NAME}"       # = MQTT_CLIENT_ID de l'edge, propre à chaque membre
DEVICE_PREFIX = f"coffee-{GROUPE_NAME}"  # machines simulées : <DEVICE_PREFIX>-01, -02, ...
print("compte", ACCOUNT_ID, "| région", REGION, "| dépôt", ROOT)
print("utilisateur", GROUPE_NAME, "| thing", THING_NAME, "| machines", f"{DEVICE_PREFIX}-01...")
assert (ROOT / "docker-compose.yml").exists(), "lancer le notebook depuis AWS/IotCore/"


def load_state():
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}


def save_state(**values):
    state = load_state() | values
    STATE_FILE.write_text(json.dumps(state, indent=2))
    return state

compte 352909266144 | région us-east-1 | dépôt /home/aydong/TSM-CloudSys/cloud_project_2_grp10
utilisateur GRP10 | thing edge-GRP10 | machines coffee-GRP10-01...


## 1. Thing

Un *thing* représente un objet dans le registre IoT Core. Ici l'objet connecté est **l'edge**

In [37]:
try:
    thing = iot.describe_thing(thingName=THING_NAME)
    print("thing existant :", thing["thingArn"])
except iot.exceptions.ResourceNotFoundException:
    thing = iot.create_thing(thingName=THING_NAME)
    print("thing créé :", thing["thingArn"])

thing créé : arn:aws:iot:us-east-1:352909266144:thing/edge-GRP10


## 2. Certificat X.509 et CA racine

IoT Core authentifie les clients par **TLS mutuel** : l'edge présente un certificat signé par AWS, et vérifie le serveur avec la CA racine Amazon.

- `edge.crt` / `edge.key` : certificat et clé privée de l'edge. **AWS ne fournit la clé privée qu'une seule fois**, à la création : elle est écrite directement dans `certs/`.
- `ca.crt` : `AmazonRootCA1.pem`.

Ce sont les noms que `edge.py` lit déjà (`MQTT_CA`, `MQTT_CERT`, `MQTT_KEY`), via le volume `./certs:/certs`. Les éventuels certificats de test locaux (`gen_certs.sh`) sont d'abord sauvegardés dans `certs/local-backup/`.

In [38]:
import urllib.request

state = load_state()
cert_ok = False
if "cert_arn" in state and (CERTS / "edge.key").exists():
    try:
        desc = iot.describe_certificate(certificateId=state["cert_id"])["certificateDescription"]
        cert_ok = desc["status"] == "ACTIVE"
    except iot.exceptions.ResourceNotFoundException:
        pass

if cert_ok:
    print("certificat existant :", state["cert_arn"])
else:
    # Sauvegarde des certificats de test locaux avant de les remplacer.
    backup = CERTS / "local-backup"
    for name in ("ca.crt", "edge.crt", "edge.key"):
        if (CERTS / name).exists():
            backup.mkdir(exist_ok=True)
            shutil.move(CERTS / name, backup / name)
            print("sauvegardé :", backup / name)

    cert = iot.create_keys_and_certificate(setAsActive=True)
    CERTS.mkdir(exist_ok=True)
    (CERTS / "edge.crt").write_text(cert["certificatePem"])
    (CERTS / "edge.key").write_text(cert["keyPair"]["PrivateKey"])
    (CERTS / "edge.key").chmod(0o600)  # le conteneur tourne en root, il peut la lire
    state = save_state(cert_arn=cert["certificateArn"], cert_id=cert["certificateId"])
    print("certificat créé :", state["cert_arn"])

urllib.request.urlretrieve("https://www.amazontrust.com/repository/AmazonRootCA1.pem", CERTS / "ca.crt")
print("CA racine :", CERTS / "ca.crt")

certificat existant : arn:aws:iot:us-east-1:352909266144:cert/f7384f32afb67e27d5101379c4369ccbe3a72b162be4572d800210c0abf39326
CA racine : /home/aydong/TSM-CloudSys/cloud_project_2_grp10/certs/ca.crt


## 3. Policy IoT

Une policy IoT dit ce que le porteur du certificat a le droit de faire. Principe du moindre privilège :

- `iot:Connect` uniquement avec l'ID client = nom du thing (`${iot:Connection.Thing.ThingName}`) ;
- `iot:Publish` uniquement sur `coffee/*/batch`.

Elle est attachée au certificat, et le certificat est attaché au thing (ce lien rend la variable `ThingName` utilisable).

In [39]:
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": "iot:Connect",
            "Resource": f"arn:aws:iot:{REGION}:{ACCOUNT_ID}:client/${{iot:Connection.Thing.ThingName}}",
        },
        {
            "Effect": "Allow",
            "Action": "iot:Publish",
            "Resource": f"arn:aws:iot:{REGION}:{ACCOUNT_ID}:topic/{TOPIC_PREFIX}/*/batch",
        },
    ],
}

try:
    iot.create_policy(policyName=POLICY_NAME, policyDocument=json.dumps(policy_document))
    print("policy créée :", POLICY_NAME)
except iot.exceptions.ResourceAlreadyExistsException:
    # Nouvelle version si le document a changé (5 versions max : on supprime les anciennes).
    current = json.loads(iot.get_policy(policyName=POLICY_NAME)["policyDocument"])
    if current != policy_document:
        for v in iot.list_policy_versions(policyName=POLICY_NAME)["policyVersions"]:
            if not v["isDefaultVersion"]:
                iot.delete_policy_version(policyName=POLICY_NAME, policyVersionId=v["versionId"])
        iot.create_policy_version(policyName=POLICY_NAME, policyDocument=json.dumps(policy_document),
                                  setAsDefault=True)
        print("policy mise à jour :", POLICY_NAME)
    else:
        print("policy existante :", POLICY_NAME)

cert_arn = load_state()["cert_arn"]
iot.attach_policy(policyName=POLICY_NAME, target=cert_arn)
iot.attach_thing_principal(thingName=THING_NAME, principal=cert_arn)
print("policy → certificat → thing : OK")

policy existante : coffee-edge-policy-GRP10
policy → certificat → thing : OK


## 4. Endpoint MQTT et `.env`

Chaque compte a son propre endpoint (`xxxx-ats.iot.<région>.amazonaws.com`). L'edge s'y connecte en `mqtts://` sur le port 8883.

La cellule met à jour `.env` à la racine (les autres variables sont conservées) :
- `MQTT_URL` pointe sur IoT Core ;
- `MQTT_USERNAME` / `MQTT_PASSWORD` sont commentés : IoT Core n'authentifie que par certificat ;
- `MQTT_CLIENT_ID` et `DEVICE_PREFIX` sont propres à chaque membre (voir étape 0) ;
- `BATCH_MAX_ROWS` limite le nombre de mesures par device dans un lot, chaque device étant envoyé dans son propre message (IoT Core refuse les messages de plus de 128 KB, ce qui arriverait au premier envoi après une longue coupure réseau).

In [40]:
ENDPOINT = iot.describe_endpoint(endpointType="iot:Data-ATS")["endpointAddress"]
print("endpoint :", ENDPOINT)

updates = {
    "MQTT_URL": f"mqtts://{ENDPOINT}:8883",
    "MQTT_CLIENT_ID": THING_NAME,
    "DEVICE_PREFIX": DEVICE_PREFIX,
    "MQTT_TOPIC_PREFIX": TOPIC_PREFIX,
    "BATCH_MAX_ROWS": str(BATCH_MAX_ROWS),
}
disabled = {"MQTT_USERNAME", "MQTT_PASSWORD"}

lines = ENV_FILE.read_text().splitlines() if ENV_FILE.exists() else []
out = []
for line in lines:
    key = line.split("=", 1)[0].strip()
    if key in updates:
        out.append(f"{key}={updates.pop(key)}")
    elif key in disabled:
        out.append(f"# {line}  (inutile avec IoT Core)")
    else:
        out.append(line)
out += [f"{k}={v}" for k, v in updates.items()]
ENV_FILE.write_text("\n".join(out) + "\n")
print(ENV_FILE.read_text())

endpoint : a115j3mq7h1g0g-ats.iot.us-east-1.amazonaws.com
# MQTT Configuration
MQTT_URL=mqtts://a115j3mq7h1g0g-ats.iot.us-east-1.amazonaws.com:8883
# MQTT_USERNAME=user  (inutile avec IoT Core)
# MQTT_PASSWORD=pass  (inutile avec IoT Core)
MQTT_CLIENT_ID=edge-GRP10

# Number of devices to simulate
NUM_DEVICES=10

# Device simulation settings
BATCH_INTERVAL_S=15
WATCHDOG_TIMEOUT_S=30
MQTT_TOPIC_PREFIX=coffee-GRP10
BATCH_MAX_ROWS=500
DEVICE_PREFIX=coffee-GRP10



## 5. Connexion

In [28]:
import ssl
import threading

import paho.mqtt.client as mqtt

connected = threading.Event()
result = {}


def on_connect(client, userdata, flags, reason_code, properties):
    result["rc"] = reason_code
    connected.set()


client = mqtt.Client(callback_api_version=mqtt.CallbackAPIVersion.VERSION2, client_id=THING_NAME)
client.tls_set(ca_certs=str(CERTS / "ca.crt"), certfile=str(CERTS / "edge.crt"),
               keyfile=str(CERTS / "edge.key"), tls_version=ssl.PROTOCOL_TLS_CLIENT)
client.on_connect = on_connect
client.connect(ENDPOINT, 8883, keepalive=30)
client.loop_start()
try:
    assert connected.wait(10), "pas de réponse du broker (réseau / endpoint ?)"
    assert not result["rc"].is_failure, f"connexion refusée : {result['rc']}"
    print("connecté à IoT Core")

    test_device = f"{DEVICE_PREFIX}-test"
    body = json.dumps({"edge_id": THING_NAME, "device_id": test_device, "sent_at": int(time.time() * 1000),
                       "readings": [], "test": True})
    info = client.publish(f"{TOPIC_PREFIX}/{test_device}/batch", body, qos=1)
    info.wait_for_publish(timeout=10)
    assert info.is_published(), "pas de PUBACK : la policy refuse sans doute la publication"
    print("publication acquittée (PUBACK) sur", f"{TOPIC_PREFIX}/{test_device}/batch")
finally:
    client.loop_stop()
    client.disconnect()

connecté à IoT Core
publication acquittée (PUBACK) sur coffee-GRP10/coffee-nathan-coudray-test/batch


## 6. Rôle IAM pour les logs IoT Core

IoT Core n'écrit dans CloudWatch Logs qu'en endossant un rôle IAM. On crée `ROLE_NAME` (confiance : `iot.amazonaws.com`) avec la policy en ligne `iot-logs`, limitée au groupe `AWSIotLogsV2`, le seul qu'IoT Core utilise.

Les logs de la Lambda (`/aws/lambda/GRP10-SdatawayLambdaIngestor`, rétention 365 jours) sont gérés par `AWS/LambdaIngestor/deploy_lambda_ingestor.ipynb` : rien à faire ici.

In [ ]:
# Nom imposé par IoT Core, non configurable : il écrit toujours dans ce groupe, un autre nom ne reçoit rien.
IOT_LOG_GROUP = "AWSIotLogsV2"
log_group_arn = f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:{IOT_LOG_GROUP}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "iot.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
logs_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents",
                       "logs:PutRetentionPolicy", "logs:DescribeLogStreams"],
            "Resource": [log_group_arn, f"{log_group_arn}:*"],
        }
    ],
}

try:
    role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print("rôle existant :", role_arn)
except iam.exceptions.NoSuchEntityException:
    role_arn = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
                               Description=f"IoT Core -> CloudWatch Logs ({GROUPE_NAME})")["Role"]["Arn"]
    print("rôle créé :", role_arn)

# put_role_policy remplace la policy en ligne : relancer la cellule applique les changements.
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="iot-logs", PolicyDocument=json.dumps(logs_policy))
print("policy iot-logs →", IOT_LOG_GROUP, ": OK")

## 7. Logs IoT Core dans CloudWatch

Le logging IoT Core se règle à deux niveaux :
- **options globales** (`set_v2_logging_options`) : le rôle utilisé et le niveau par défaut, **pour tout le compte et la région**. Le compte étant partagé avec d'autres groupes, on laisse le niveau par défaut à `DISABLED`, et on ne touche à rien si quelqu'un a déjà configuré le logging (son rôle écrira aussi nos logs) ;
- **niveau par ressource** (`set_v2_logging_level`) : on active les logs uniquement pour notre client MQTT `THING_NAME`.

Niveaux : `ERROR` < `WARN` < `INFO` < `DEBUG`. `WARN` suffit en fonctionnement normal (connexions refusées, publications rejetées par la policy). `INFO` journalise en plus chaque connexion et chaque publication : utile pour déboguer, mais coûteux en continu (une ligne par lot et par machine).

Les logs arrivent dans `AWSIotLogsV2` (nom imposé par IoT Core, commun à tout le compte). Le groupe est créé ici avec une rétention de `LOG_RETENTION_DAYS` = 365 jours (cahier des charges : logs conservés 1 an).

In [39]:
IOT_LOG_LEVEL = "WARN" 

# Groupe créé à l'avance pour fixer la rétention (sinon IoT Core le crée sans expiration).
try:
    logs.create_log_group(logGroupName=IOT_LOG_GROUP)
    print("groupe de logs créé :", IOT_LOG_GROUP)
except logs.exceptions.ResourceAlreadyExistsException:
    print("groupe de logs existant :", IOT_LOG_GROUP)
logs.put_retention_policy(logGroupName=IOT_LOG_GROUP, retentionInDays=LOG_RETENTION_DAYS)
print("rétention :", LOG_RETENTION_DAYS, "jours")

try:
    options = iot.get_v2_logging_options()
    print("logging déjà configuré sur le compte, conservé :", options["roleArn"], options["defaultLogLevel"])
except iot.exceptions.NotConfiguredException:
    # Un rôle tout juste créé met quelques secondes à pouvoir être endossé par IoT Core.
    for _ in range(12):
        try:
            iot.set_v2_logging_options(roleArn=role_arn, defaultLogLevel="DISABLED")
            break
        except iot.exceptions.InvalidRequestException:
            time.sleep(5)
    else:
        raise RuntimeError(f"IoT Core n'arrive pas à endosser le rôle {ROLE_NAME}")
    print("logging activé sur le compte (niveau par défaut DISABLED), rôle", ROLE_NAME)

iot.set_v2_logging_level(logTarget={"targetType": "CLIENT_ID", "targetName": THING_NAME}, logLevel=IOT_LOG_LEVEL)
print(f"logs {IOT_LOG_LEVEL} activés pour le client {THING_NAME} → {IOT_LOG_GROUP}")

groupe de logs existant : AWSIotLogsV2
rétention : 365 jours
logging déjà configuré sur le compte, conservé : arn:aws:iam::352909266144:role/coffee-iot-role-GRP10 DISABLED
logs WARN activés pour le client edge-GRP10 → AWSIotLogsV2


## 8. Vérification des logs

Affiche les derniers événements IoT Core de notre client sur la dernière heure. En `WARN`, une liste vide signifie qu'il n'y a pas eu d'erreur. 

Dans la console : CloudWatch → Log groups → `AWSIotLogsV2`, ou Logs Insights :

```
fields @timestamp, eventType, logLevel, status, reason
| filter clientId = "edge-GRP10"
| sort @timestamp desc
```

In [40]:
since = int((time.time() - 3600) * 1000)
events = logs.filter_log_events(logGroupName=IOT_LOG_GROUP, startTime=since,
                                filterPattern=f'{{ $.clientId = "{THING_NAME}" }}', limit=20)["events"]
for event in events:
    entry = json.loads(event["message"])
    print(entry.get("timestamp"), entry.get("logLevel"), entry.get("eventType"), entry.get("status"),
          entry.get("reason", ""))
print(len(events), "événement(s) pour", THING_NAME, "sur la dernière heure")

2026-10-27 09:58:38.907 INFO RuleMatch Success 
2026-10-27 09:58:38.922 INFO RuleExecution Success 
2026-10-27 09:58:39.018 INFO RuleMatch Success 
2026-10-27 09:58:39.030 INFO RuleExecution Success 
2026-10-27 09:58:39.120 INFO RuleMatch Success 
2026-10-27 09:58:39.142 INFO RuleExecution Success 
2026-10-27 09:58:39.226 INFO RuleMatch Success 
2026-10-27 09:58:39.249 INFO RuleExecution Success 
2026-10-27 09:58:39.335 INFO RuleMatch Success 
2026-10-27 09:58:39.351 INFO RuleExecution Success 
2026-10-27 09:58:39.549 INFO RuleMatch Success 
2026-10-27 09:58:39.575 INFO RuleExecution Success 
2026-10-27 09:58:39.658 INFO RuleMatch Success 
2026-10-27 09:58:39.695 INFO RuleExecution Success 
2026-10-27 09:58:39.766 INFO Publish-In Success 
2026-10-27 09:58:39.876 INFO RuleMatch Success 
2026-10-27 09:58:39.880 INFO Publish-In Success 
2026-10-27 09:58:39.897 INFO RuleExecution Success 
2026-10-27 09:58:39.983 INFO RuleMatch Success 
2026-10-27 09:58:39.985 INFO Publish-In Success 
20 év

## 9. Nettoyage (optionnel)

Le compte est partagé : le nettoyage est en deux niveaux.

- `CLEANUP = True` : supprime uniquement **vos** ressources, c'est-à-dire votre certificat (détaché, désactivé puis supprimé), votre thing `edge-<utilisateur>` et le niveau de logs de votre client MQTT. Sans risque pour les autres membres.
- `CLEANUP_SHARED = True` : supprime en plus les ressources **communes** (règle d'ingestion, permission sur la Lambda, policy, rôle). Casse le pipeline de tout le groupe : à ne faire qu'en fin de projet, d'un commun accord.

Les groupes de logs et les fichiers de `certs/` sont conservés.

In [ ]:
CLEANUP = False
CLEANUP_SHARED = False


def ignore_missing(fn, **kwargs):
    try:
        fn(**kwargs)
    except ClientError as exc:
        print("  ignoré :", exc.response["Error"]["Code"])


if CLEANUP:
    state = load_state()
    if "cert_arn" in state:
        ignore_missing(iot.detach_thing_principal, thingName=THING_NAME, principal=state["cert_arn"])
        ignore_missing(iot.detach_policy, policyName=POLICY_NAME, target=state["cert_arn"])
        ignore_missing(iot.update_certificate, certificateId=state["cert_id"], newStatus="INACTIVE")
        ignore_missing(iot.delete_certificate, certificateId=state["cert_id"], forceDelete=True)
    ignore_missing(iot.delete_thing, thingName=THING_NAME)
    ignore_missing(iot.delete_v2_logging_level, targetType="CLIENT_ID", targetName=THING_NAME)
    STATE_FILE.unlink(missing_ok=True)
    print("vos ressources sont supprimées")

if CLEANUP_SHARED:
    ignore_missing(iot.delete_topic_rule, ruleName=RULE_NAME)
    ignore_missing(lambda_.remove_permission, FunctionName=LAMBDA_NAME, StatementId=f"iot-{RULE_NAME}")
    try:
        for v in iot.list_policy_versions(policyName=POLICY_NAME)["policyVersions"]:
            if not v["isDefaultVersion"]:
                iot.delete_policy_version(policyName=POLICY_NAME, policyVersionId=v["versionId"])
    except iot.exceptions.ResourceNotFoundException:
        pass
    # Refusé (DeleteConflict) tant qu'un certificat d'un autre membre utilise la policy.
    ignore_missing(iot.delete_policy, policyName=POLICY_NAME)
    ignore_missing(iam.delete_role_policy, RoleName=ROLE_NAME, PolicyName="iot-logs")
    ignore_missing(iam.delete_role, RoleName=ROLE_NAME)
    print("ressources communes supprimées")